# 🖥️ Local AI Developer Assistant
**Day 10 · AI Application Development Bootcamp**

This lab builds a developer assistant that runs entirely on **your own machine** using Ollama — no cloud API, no internet needed after setup.

---

### How to use this notebook
- Run cells **in order** — later cells depend on earlier ones
- Cells marked `# ✏️ YOUR TURN` have gaps for you to fill in
- Write your reflection in the **📝 Reflection** section at the end

### Sections at a glance

| Part | Topic | Type |
|------|-------|------|
| A | Connect to Ollama | Run & read |
| B | Core generation function | Fill in |
| C | Prompt templates for developer tasks | Fill in |
| D | Multi-turn conversation with `/api/chat` | Fill in |
| E | Task classifier and router | Fill in |
| F | Multi-model benchmark | Fill in |
| G | Optional UI (Streamlit or Gradio) | Open |
| H | Reflection | Write |

### Before you start
Make sure Ollama is running and at least one model is pulled (see handout Section 6):
```bash
ollama serve
```
Suggested models — pick what fits your hardware:

| Model | Min RAM | Notes |
|-------|---------|-------|
| `qwen2.5:1.5b` | 4 GB | Today's default — works on any laptop |
| `llama3.2:1b` | 4 GB | Alternative — equally valid |
| `llama3.2:3b` | 6 GB | Better quality, still fast |
| `qwen3:4b` | 8 GB | Noticeably better if hardware allows |


---
## ⚙️ Setup — Imports


In [18]:
# If needed: !pip install requests pandas

import requests
import time
import pandas as pd
from typing import Dict, Any, List

OLLAMA_URL = "http://localhost:11434"

# ✏️ Change this to whichever model you pulled.
# The benchmark (Part F) will compare this against a second model.
MODEL_NAME = "qwen2.5:1.5b"

print(f"Using model: {MODEL_NAME}")
print(f"Ollama URL : {OLLAMA_URL}")


Using model: qwen2.5:1.5b
Ollama URL : http://localhost:11434


---
## Part A — Connect to Ollama

Before writing any prompts, confirm that Ollama is running and the right models are installed.
If this cell fails, run `ollama serve` in a separate terminal and try again.


In [19]:
def check_ollama() -> bool:
    """Return True if Ollama is reachable and list installed models."""
    try:
        response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
        response.raise_for_status()
        models = response.json().get("models", [])
        print("✅ Ollama is running.")
        print(f"   Installed models ({len(models)}):")
        for m in models:
            print("   -", m.get("name"))
        return True
    except Exception as e:
        print("❌ Could not connect to Ollama.")
        print("   Run `ollama serve` in a separate terminal, then try again.")
        print("   Error:", e)
        return False

check_ollama()


✅ Ollama is running.
   Installed models (7):
   - qwen2.5:1.5b
   - qwen3:4b
   - qwen3:0.6b
   - gemma3:4b
   - nomic-embed-text:latest
   - llama3.2:3b
   - gemma4:e2b


True

---
## Part B — Core Generation Function

This is the **only place in the whole lab where a network call is made**. Every prompt template, classifier, and benchmark you build in later sections calls this one function.

It uses `/api/generate` — one prompt in, one response out, no memory of previous turns. You will add multi-turn memory in Part D.

**Why `temperature=0.2`?**  
Low temperature means the model picks the most likely next token rather than sampling creatively. For developer tasks — debugging, explanation, test generation — you want **consistent, factual answers**, not creative variation. Try higher values (0.7+) only for open-ended tasks.

**What you should see after filling this in:**  
A short direct answer to the test prompt, printed cleanly.


In [28]:
def ask_ollama(
    prompt: str,
    model: str = MODEL_NAME,
    temperature: float = 0.2
) -> str:
    """
    Send a single prompt to a local Ollama model and return the response text.

    Endpoint: POST /api/generate
    Request body: {model, prompt, stream, options: {temperature}}
    Response: response.json()["response"]
    """
    url = f"{OLLAMA_URL}/api/generate"

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }

    response = requests.post(
        url,
        json=payload,
        timeout=1200
    )

    response.raise_for_status()

    return response.json().get("response", "").strip()

---
## Part C — Prompt Templates for Developer Tasks

A prompt template wraps the user's raw input (code, error message, etc.) in a structured instruction that produces consistent, usable output from the model.

**Key principles for developer prompts:**
- Give the model a clear role: `"You are a debugging assistant."`
- Wrap code in a fenced block so the model treats it as code, not prose
- Be explicit about output format — ask for numbered steps, not open-ended paragraphs
- Always ask for a corrected or improved version — makes the output actionable

Implement at least **three** of the four templates below.


In [29]:
def make_code_explainer_prompt(code_snippet: str) -> str:
    """
    Create a prompt that explains code to a beginner.
    """
    return f"""
You are a helpful programming tutor.

Review the following Python code:

```python
{code_snippet}
```

Answer using numbered sections:

1. Explain what the code does.
2. Explain the important lines.
3. Identify any bugs, risks, or confusing parts.
4. Provide a corrected or improved version if needed.

Use simple language suitable for a beginner.
""".strip()


def make_debug_prompt(error_message: str, code_snippet: str = "") -> str:
    """
    Create a prompt that helps debug Python code.
    """
    if code_snippet.strip():
        code_section = f"""
```python
{code_snippet}
```
"""
    else:
        code_section = "No code snippet was provided."

    return f"""
You are a helpful Python debugging assistant.

Error message:

```text
{error_message}
```

Code:

{code_section}

Answer using numbered sections:

1. Explain the most likely cause of the error.
2. Identify the line or section causing the problem.
3. Provide a simple fix.
4. Provide a safer rewritten version of the code.
5. Briefly explain the changes.

Use simple language suitable for a beginner.
""".strip()


def make_testcase_prompt(code_snippet: str) -> str:
    """
    Create a prompt that asks for test-case suggestions.
    """
    return f"""
You are a software testing assistant.

Review the following Python code:

```python
{code_snippet}
```

Suggest suitable test cases.

For each test case, provide:

1. Input
2. Expected output
3. Why the test matters

Include normal test cases and these edge cases where applicable:

- Empty input
- None
- Wrong data type
- Boundary values

Present the test cases clearly.
""".strip()


def make_improvement_prompt(code_snippet: str) -> str:
    """
    Create a prompt that reviews and improves Python code.
    """
    return f"""
You are an experienced Python code reviewer.

Review the following Python code:

```python
{code_snippet}
```

Answer using numbered sections:

1. Identify readability issues.
2. Identify missing error handling.
3. Provide performance or efficiency notes.
4. Provide a cleaner rewritten version.
5. Briefly explain each change.

Keep the improved code simple and readable.
""".strip()

### C2 — Try your templates on a real example

Uncomment each block after implementing the corresponding template. You should see the model produce a structured response for each task type.


In [30]:
sample_code = '''
def divide_numbers(a, b):
    return a / b

print(divide_numbers(10, 0))
'''

sample_error = "ZeroDivisionError: division by zero"

# ── Explanation ───────────────────────────────────────────────────────────────
print("=== 📖 EXPLANATION ===")
print(ask_ollama(make_code_explainer_prompt(sample_code)))

# ── Debug ─────────────────────────────────────────────────────────────────────
print("\n=== 🐛 DEBUG ===")
print(ask_ollama(make_debug_prompt(sample_error, sample_code)))

# ── Test cases ────────────────────────────────────────────────────────────────
print("\n=== 🧪 TEST CASES ===")
print(ask_ollama(make_testcase_prompt(sample_code)))

# ── Improvement ───────────────────────────────────────────────────────────────
print("\n=== ✨ IMPROVEMENT ===")
print(ask_ollama(make_improvement_prompt(sample_code)))


=== 📖 EXPLANATION ===
### 1. What the Code Does

The Python code defines a function named `divide_numbers` that takes two arguments: `a` and `b`. The function then divides `a` by `b` using the `/` operator, which is equivalent to performing integer division (i.e., truncating any decimal part) if both inputs are integers. Finally, it returns the result of this division.

### 2. Important Lines

- **Line 1:** `def divide_numbers(a, b):`
  - This line defines a function named `divide_numbers` that takes two parameters: `a` and `b`.
  
- **Line 3:** `return a / b`
  - This line calls the function with arguments `a` and `b`, performs division on them using `/`, and returns the result.

### 3. Bugs, Risks, or Confusing Parts

- The code currently divides by zero when it encounters an input of `0`. This is not a good practice because dividing any number by zero results in a `ZeroDivisionError`.
  
- There are no obvious bugs, risks, or confusing parts in the current implementation.

### 4. Co

---
## Part D — Multi-turn Conversation with `/api/chat`

So far you have used `/api/generate` — one prompt, one response, no memory of previous turns.

`/api/chat` accepts a `messages` list (same format as OpenAI's API). Each call sends the **full conversation history**, so the model can refer back to earlier messages.

This matters for a developer assistant: a user might say *"explain this function"*, then follow up with *"now add error handling to it"* — and the model needs to know what *"it"* refers to.

**Message format:**
```python
[
    {"role": "system",    "content": "..."},   # optional — sets the model's behaviour
    {"role": "user",      "content": "..."},   # first user message
    {"role": "assistant", "content": "..."},   # model's reply — append this after each turn
    {"role": "user",      "content": "..."},   # next user message
]
```

**What you should see:** Turn 1 explains the function. Turn 2 produces a rewritten version with error handling — and references what was discussed in Turn 1, without you repeating the code.


In [31]:
from typing import List, Dict
import requests


def chat_with_ollama(
    messages: List[Dict[str, str]],
    model: str = MODEL_NAME,
    temperature: float = 0.2
) -> str:
    """
    Multi-turn chat using /api/chat.
    Returns the model's latest reply as a string.

    Endpoint: POST /api/chat
    Request body: {model, messages, stream: False, options: {temperature}}
    Response: response.json()["message"]["content"]
    """
    url = f"{OLLAMA_URL}/api/chat"

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }

    response = requests.post(
        url,
        json=payload,
        timeout=120
    )

    response.raise_for_status()

    return response.json().get("message", {}).get("content", "").strip()


# ── Two-turn demo ──────────────────────────────────────────────────────────────
conversation = [
    {
        "role": "system",
        "content": "You are a helpful Python tutor."
    },
    {
        "role": "user",
        "content": f"Explain this function:\n```python\n{sample_code}\n```"
    }
]

reply1 = chat_with_ollama(conversation)

print("Turn 1 — Explanation:")
print(reply1)


# Append the model's first reply to the conversation history
conversation.append({
    "role": "assistant",
    "content": reply1
})

# Add the user's follow-up question
conversation.append({
    "role": "user",
    "content": "Now rewrite it with proper error handling."
})

reply2 = chat_with_ollama(conversation)

print("\nTurn 2 — Rewrite:")
print(reply2)

Turn 1 — Explanation:
The provided Python function `divide_numbers` takes two arguments: `a` and `b`. The function returns the result of dividing `a` by `b`.

Here's a step-by-step breakdown:

1. **Function Definition**: 
   ```python
   def divide_numbers(a, b):
       return a / b
   ```
   - This defines a function named `divide_numbers`.
   - It takes two parameters: `a` and `b`.
   - The function body is the line that performs the division operation using `/`.

2. **Division Operation**:
   ```python
   return a / b
   ```
   - This line computes the result of dividing `a` by `b`.
   - If `b` is zero, this will raise a `ZeroDivisionError`, which means you cannot divide by zero.

3. **Function Call**:
   ```python
   print(divide_numbers(10, 0))
   ```
   - This calls the `divide_numbers` function with arguments `10` and `0`.
   - The division operation will result in a `ZeroDivisionError`, which is caught by Python's exception handling mechanism.
   - When you run this code, it pr

---
## Part E — Task Classifier and Router

A real developer assistant should not ask the user to pick a task from a dropdown. It reads the request, decides what kind of task it is, selects the right prompt template automatically, and responds.

This section builds that in two steps:
1. **`classify_task()`** — reads the user's request and returns a task type string
2. **`developer_assistant()`** — uses the task type to select a prompt template and return the answer

The classifier is keyword-based — `if`/`elif` on lowercased text. This is intentional: you don't need machine learning for well-defined categories with reliable keywords. The important design point is **separating classification from generation** — the classifier never calls the model, and the generator doesn't need to know about classification.

**What you should see:** each test request is routed to a different task type and produces the right kind of response.


In [32]:
def classify_task(user_request: str) -> str:
    """
    Classify a free-text developer request into one of four task types.

    Returns one of:
    - "debug"
    - "test"
    - "improve"
    - "explain"
    """
    text = user_request.lower()

    debug_keywords = [
        "error",
        "bug",
        "fix",
        "debug",
        "exception",
        "traceback",
        "fails",
    ]

    test_keywords = [
        "test",
        "case",
        "assert",
        "pytest",
        "unittest",
        "coverage",
    ]

    improve_keywords = [
        "improve",
        "refactor",
        "clean",
        "optimise",
        "optimize",
        "rewrite",
        "review",
    ]

    if any(keyword in text for keyword in debug_keywords):
        return "debug"

    elif any(keyword in text for keyword in test_keywords):
        return "test"

    elif any(keyword in text for keyword in improve_keywords):
        return "improve"

    else:
        return "explain"


def developer_assistant(
    user_request: str,
    code_snippet: str = "",
    error_message: str = ""
) -> Dict[str, str]:
    """
    Classify the request, select the correct prompt template,
    send it to Ollama, and return the task type and answer.
    """
    task = classify_task(user_request)

    if task == "debug":
        prompt = make_debug_prompt(
            error_message=error_message,
            code_snippet=code_snippet,
        )

    elif task == "test":
        prompt = make_testcase_prompt(code_snippet)

    elif task == "improve":
        prompt = make_improvement_prompt(code_snippet)

    else:
        prompt = make_code_explainer_prompt(code_snippet)

    answer = ask_ollama(prompt)

    return {
        "task": task,
        "answer": answer,
    }


# ── Test the router on four requests ───────────────────────────────────────────
requests_to_test = [
    ("Please debug this", sample_code, sample_error),
    ("Suggest test cases", sample_code, ""),
    ("Refactor this function", sample_code, ""),
    ("What does this do?", sample_code, ""),
]

for req, code_s, err in requests_to_test:
    result = developer_assistant(req, code_s, err)

    print(f"Request  : {req}")
    print(f"Detected : {result['task']}")
    print(f"Answer   : {result['answer'][:250]}...")
    print()

Request  : Please debug this
Detected : debug
Answer   : 1. The most likely cause of the error is that one of the numbers being divided (in this case, `a` and `b`) is zero. When you try to divide by zero, Python throws a ZeroDivisionError because division by zero is undefined in mathematics.

2. The line c...

Request  : Suggest test cases
Detected : test
Answer   : Sure, let's review the Python code provided and suggest suitable test cases.

### Code Review:
```python
def divide_numbers(a, b):
    return a / b

print(divide_numbers(10, 0))
```

### Test Cases:

#### Normal Test Cases:
- **Input:** `a = 10`, `b ...

Request  : Refactor this function
Detected : improve
Answer   : ### 1. Readability Issues

The current function `divide_numbers` is straightforward but lacks readability, especially when it comes to error handling for division by zero. This can lead to confusion or bugs in production environments where such error...

Request  : What does this do?
Detected : explain
Answer  

---
## Part F — Multi-Model Benchmark

One of the key questions in local AI is: **how much quality do you give up by using a smaller model, and is the speed gain worth it?**

This section makes that concrete: run the same three prompts through two different model sizes and compare results side by side.

**What to think about while reading the results:**
- Is the quality difference noticeable on the simple task? What about the complex one?
- At which task does the smaller model fall apart first?
- If you were building a real coding assistant, which model would you choose — and why?

**What you should see:** a DataFrame with elapsed times and response lengths per model,
plus two full answers side-by-side for the complex task.


In [33]:
import time
import pandas as pd
from typing import Dict, Any


# Both models must already appear when you run: ollama list
MODEL_A = "qwen3:4b"
MODEL_B = "qwen2.5:1.5b"


def benchmark_prompt(prompt: str, model: str) -> Dict[str, Any]:
    """
    Run one prompt through one model and record timing and output length.
    """
    start_time = time.perf_counter()

    answer = ask_ollama(
        prompt,
        model=model
    )

    end_time = time.perf_counter()
    elapsed = end_time - start_time

    return {
        "model": model,
        "prompt_chars": len(prompt),
        "response_chars": len(answer),
        "elapsed_seconds": elapsed,
        "answer": answer,
    }


benchmark_prompts = [
    (
        "Simple",
        "Explain recursion in Python in one short paragraph."
    ),

    (
        "Debug",
        f"""
Find the bug in this code and explain why it fails:

```python
{sample_code}
```
""".strip()
    ),

    (
        "Complex",
        """
Write a Python function that reads a CSV file, filters rows where a given
column exceeds a threshold, and returns the result as a list of dictionaries.

Include:
1. Type hints
2. A docstring
3. Error handling for missing files
""".strip()
    ),
]


# Run all three prompts through both models
results = []

for label, prompt in benchmark_prompts:
    for model in [MODEL_A, MODEL_B]:
        print(f"Running {label} task with {model}...")

        row = benchmark_prompt(
            prompt=prompt,
            model=model
        )

        row["task"] = label
        results.append(row)


# Convert the results into a DataFrame
df = pd.DataFrame(results)

comparison_df = df[
    [
        "task",
        "model",
        "elapsed_seconds",
        "prompt_chars",
        "response_chars",
    ]
].copy()

comparison_df["elapsed_seconds"] = comparison_df[
    "elapsed_seconds"
].round(2)

print("\n=== BENCHMARK RESULTS ===")
display(comparison_df)


# Display the complete answers for the Complex task
complex_answers = df[df["task"] == "Complex"][
    ["model", "answer"]
]

print("\n=== COMPLEX TASK ANSWERS ===")

for _, row in complex_answers.iterrows():
    print("\n" + "=" * 80)
    print(f"MODEL: {row['model']}")
    print("=" * 80)
    print(row["answer"])

Running Simple task with qwen3:4b...
Running Simple task with qwen2.5:1.5b...
Running Debug task with qwen3:4b...
Running Debug task with qwen2.5:1.5b...
Running Complex task with qwen3:4b...
Running Complex task with qwen2.5:1.5b...

=== BENCHMARK RESULTS ===


,task,model,elapsed_seconds,prompt_chars,response_chars
0,Simple,qwen3:4b,22.39,51,462
1,Simple,qwen2.5:1.5b,10.29,51,342
2,Debug,qwen3:4b,56.47,141,1974
3,Debug,qwen2.5:1.5b,14.30,141,1754
4,Complex,qwen3:4b,69.94,226,3578
5,Complex,qwen2.5:1.5b,14.94,226,2156



=== COMPLEX TASK ANSWERS ===

MODEL: qwen3:4b
To solve this problem, we need to create a Python function that reads a CSV file, filters rows where a specified column exceeds a given threshold, and returns the result as a list of dictionaries. The solution must include type hints, a docstring, and error handling for missing files.

### Approach
1. **Import Necessary Modules**: We use the `csv` module to read CSV files and handle the file operations.
2. **Error Handling for Missing Files**: The function will catch `FileNotFoundError` to handle cases where the specified CSV file does not exist.
3. **Reading and Filtering Rows**: 
   - Use `csv.DictReader` to read the CSV file, which allows us to process each row as a dictionary.
   - Convert the specified column to a float for comparison with the threshold.
   - Filter rows where the column value exceeds the threshold.
4. **Type Hints and Docstring**: The function includes detailed type hints and a docstring explaining the purpose, param

### F2 — Quality comparison

Read both model answers for the **Complex** task side by side and rate each (1 = poor, 5 = excellent).

| Task | Model A answer | Model B answer | Notes |
|------|---------------|----------------|-------|
| Simple | 5/5 | 4/5 | Both should handle the simple explanation well, but Model A is likely clearer and more detailed. |
| Debug | 5/5 | 4/5 | Both identify division by zero, but Model A generally provides a more structured explanation and safer correction. |
| Complex | 5/5 | 4/5 | Both produced working solutions. Model A gave a more detailed explanation and handled invalid row values, while Model B used broader exception handling. |


In [36]:
# ── Print full answers for the Complex task side by side ─────────────────────
# Uncomment after the benchmark cell runs successfully.

complex_prompt = benchmark_prompts[2][1]

for model in [MODEL_A, MODEL_B]:
    print(f"{'='*60}")
    print(f"  {model}")
    print(f"{'='*60}")
    print(ask_ollama(complex_prompt, model=model))
    print()


  qwen3:4b
Here's the Python function that meets all the requirements: type hints, docstring, error handling for missing files, and proper CSV processing:

```python
import csv

def filter_csv_above_threshold(file_path: str, column: str, threshold: float) -> list[dict]:
    """
    Reads a CSV file, filters rows where the given column exceeds the threshold, and returns the result as a list of dictionaries.

    Args:
        file_path (str): Path to the CSV file.
        column (str): The column name to filter by.
        threshold (float): The threshold value. Rows where the column value is greater than this threshold are included.

    Returns:
        list[dict]: A list of dictionaries, each representing a row from the CSV where the specified column exceeds the threshold.

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError: If the column is not found in the CSV header or if the conversion of the column value to float fails.

    Note: The CSV is as

---
## Part G — Optional: UI with Streamlit or Gradio

If time allows, wrap `developer_assistant()` in a simple web UI. The goal is to confirm the same local model works through a browser interface — not to build a polished product.

Save either snippet as a `.py` file and run it from the terminal.

### Option A — Streamlit

```python
# dev_assistant_app.py
# Run with: streamlit run dev_assistant_app.py

import streamlit as st
# import your functions from this notebook or copy them here

st.title("🖥️ Local AI Developer Assistant")
st.caption("Running on Ollama — no internet needed")

code_input    = st.text_area("Paste your code here", height=200)
error_input   = st.text_input("Error message (optional)")
request_input = st.text_input("What do you want?  e.g. explain this, debug this, suggest tests")

if st.button("Run assistant") and request_input:
    with st.spinner("Thinking locally..."):
        result = developer_assistant(request_input, code_input, error_input)
    st.markdown(f"**Classified as:** `{result['task']}`")
    st.markdown(result["answer"])
```

### Option B — Gradio

```python
# dev_assistant_gradio.py
# Run with: python dev_assistant_gradio.py

import gradio as gr
# import your functions from this notebook or copy them here

def run(request, code, error):
    result = developer_assistant(request, code, error)
    return f"Classified as: {result['task']}\n\n{result['answer']}"

gr.Interface(
    fn=run,
    inputs=[
        gr.Textbox(label="What do you want?"),
        gr.Textbox(label="Code (optional)", lines=8),
        gr.Textbox(label="Error message (optional)"),
    ],
    outputs=gr.Textbox(label="Answer", lines=15),
    title="Local AI Developer Assistant"
).launch()
```


---
## 📝 Reflection

Write **150–250 words** addressing the questions below. Connect your answers to what you actually observed in Parts C – F — not just what the handout says.

1. **Model choice and hardware:** which model(s) did you use, and why that size given your machine?
2. **Benchmark results:** was the speed difference between your two models significant? At which task did quality diverge most?
3. **Real-world relevance:** from the handout's examples — Samsung data breach, air-gapped systems, cost at scale — which is most relevant to a project you could imagine building, and why?
4. **When to use cloud:** when would you still choose Groq or OpenAI over a local model for a developer assistant?


*Your reflection (150–250 words):*

I used qwen3:4b as my model A and qwen2.5:1.5b as my model B through ollama. I chose these 2 models as they were small enough to run locally on my computer while still being good enough to handle the code explanation, debugging, testing and rewriting that was required for the lab today. However model A required more time and resources, to the point that I needed to increase the timeout cap, but it generally provided a more detailed answer.

The benchmark showed a noticeable speed difference especially when loading and running the larger models, the smaller model was faster, but the biggest quality difference appeared in the complex CSV task. Both models were able to produce working codes but qwen3:4b included better column validation, clearer documentation and more careful handling of invalid numeric values. The simpler explanation and debugging tasks showed less difference because both models could understand them easily.

I believe building a local LLM developer assistant that processes source code and internal documents entirely on the user’s machine. Since the data is not sent to external cloud services, sensitive company information is less likely to be exposed through third-party systems.

I would still choose Groq or OpenAI when I need stronger reasoning, better code quality, faster generation on weak hardware, or access from multiple devices. Cloud models would also be preferable for difficult debugging tasks where accuracy matters more than privacy or offline access.




